## 0. 환경

터미널에서 실행 (프로젝트 루트, PowerShell):
```powershell
uv venv
uv pip install -r requirements.txt
source .venv/Scripts/activate
```
노트북 커널은 `AIOps-Power-Consumption-Prediction` 선택.

# 공장 전력사용량 1시간 후 예측 - LSTM (baseline)

입력: 과거 24시간의 temperature / humidity / energy_relative_pct
출력: 다음 1시간의 target_energy_relative_pct (변화율)

이 노트북은 baseline 체크포인트(factory_energy_lstm.pt) 생성용이다.  
`target_time_utc` 기준 시간순으로 미리 분리된 data/train.csv, data/valid.csv, data/test.csv를 사용합니다. Validation은 epoch마다 평가하고 Early Stopping에 사용하며 Test는 최종 모델 확정 후 한 번만 평가합니다.

In [1]:
import copy
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
print(sys.executable)
print("torch:", torch.__version__)
print("DEVICE:", DEVICE)

c:\Users\panteng\Desktop\SKALA\skala-workspace\AIOps-Power-Consumption-Prediction\.venv\Scripts\python.exe
torch: 2.13.0+cpu
DEVICE: cpu


In [2]:
# 분할 파일 직접 사용 (train/valid/test 이미 분리됨, notebooks/data/)
from pathlib import Path
_cwd = Path().resolve()
DATA_DIR = _cwd / "notebooks" / "data" if (_cwd / "notebooks").exists() else _cwd / "data"
print("DATA_DIR:", DATA_DIR)
TRAIN_PATH = str(DATA_DIR / "train.csv")
VALID_PATH = str(DATA_DIR / "valid.csv")
TEST_PATH = str(DATA_DIR / "test.csv")

train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)
test_df = pd.read_csv(TEST_PATH)
for _df in (train_df, valid_df, test_df):
    _df["target_time_utc"] = pd.to_datetime(_df["target_time_utc"], utc=True)
    _df.sort_values("target_time_utc", inplace=True)
    _df.reset_index(drop=True, inplace=True)
print(train_df.shape, valid_df.shape, test_df.shape)
print(train_df[["target_time_utc", "target_energy_relative_pct"]].head())

DATA_DIR: C:\Users\panteng\Desktop\SKALA\skala-workspace\AIOps-Power-Consumption-Prediction\notebooks\data
(30366, 74) (6507, 74) (6508, 74)
            target_time_utc  target_energy_relative_pct
0 2016-06-13 07:00:00+00:00                    0.125918
1 2016-06-13 08:00:00+00:00                   -0.545804
2 2016-06-13 09:00:00+00:00                    0.260221
3 2016-06-13 10:00:00+00:00                   -0.520146
4 2016-06-13 11:00:00+00:00                   -0.243389


## 1. 24시간 × 3개 feature 구성

각 행은 `lag_24h -> ... -> lag_1h`의 24시간 window이며, timestep마다 `[temp, humidity, energy]` 3개 feature를 갖습니다.

In [3]:
features = []
for lag in range(24, 0, -1):
    features += [f"temp_F_lag_{lag}h", f"humi_pct_lag_{lag}h", f"energy_relative_pct_lag_{lag}h"]
target_col = "target_energy_relative_pct"
for name, _df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    missing = [c for c in features + [target_col] if c not in _df.columns]
    if missing:
        raise ValueError(f"[{name}] Missing columns: {missing}")
def make_xy(_df):
    X_flat = _df[features].to_numpy(dtype=np.float32)
    y = _df[target_col].to_numpy(dtype=np.float32)
    ts = _df["target_time_utc"].to_numpy()
    return X_flat.reshape(-1, 24, 3), y, ts
X_train, y_train, ts_train = make_xy(train_df)
X_valid, y_valid, ts_valid = make_xy(valid_df)
X_test, y_test, ts_test = make_xy(test_df)
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_train: (30366, 24, 3) y_train: (30366,)
X_valid: (6507, 24, 3) y_valid: (6507,)
X_test: (6508, 24, 3) y_test: (6508,)


## 2. Scaling

Scaler는 Train에만 fit합니다. Validation/Test는 transform만 합니다.

In [4]:
feature_scaler = StandardScaler()
target_scaler = StandardScaler()
feature_scaler.fit(X_train.reshape(-1, 3))
target_scaler.fit(y_train.reshape(-1, 1))

def scale_X(a):
    shape = a.shape
    return feature_scaler.transform(a.reshape(-1, 3)).reshape(shape).astype(np.float32)

X_train_s, X_valid_s, X_test_s = map(scale_X, [X_train, X_valid, X_test])
y_train_s = target_scaler.transform(y_train.reshape(-1, 1)).astype(np.float32).ravel()
y_valid_s = target_scaler.transform(y_valid.reshape(-1, 1)).astype(np.float32).ravel()
y_test_s = target_scaler.transform(y_test.reshape(-1, 1)).astype(np.float32).ravel()

In [5]:
BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train_s), torch.from_numpy(y_train_s)), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
valid_loader = DataLoader(TensorDataset(torch.from_numpy(X_valid_s), torch.from_numpy(y_valid_s)), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test_s), torch.from_numpy(y_test_s)), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

## 3. LSTM 모델

In [6]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size=3, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

model = LSTMRegressor().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model)

LSTMRegressor(
  (lstm): LSTM(3, 64, num_layers=2, batch_first=True, dropout=0.2)
  (head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)


## 4. Train → Validation 반복 / Early Stopping

In [7]:
def evaluate_loader(model, loader):
    model.eval(); preds=[]; actuals=[]
    with torch.no_grad():
        for xb, yb in loader:
            pred = model(xb.to(DEVICE)).cpu().numpy()
            preds.append(pred); actuals.append(yb.numpy())
    pred_s=np.concatenate(preds); actual_s=np.concatenate(actuals)
    pred=target_scaler.inverse_transform(pred_s.reshape(-1,1)).ravel()
    actual=target_scaler.inverse_transform(actual_s.reshape(-1,1)).ravel()
    return np.sqrt(mean_squared_error(actual,pred)), mean_absolute_error(actual,pred), pred, actual

EPOCHS=50; PATIENCE=7
best_val_rmse=float("inf"); best_state=None; patience=0; history=[]

for epoch in range(1,EPOCHS+1):
    model.train()
    for xb,yb in train_loader:
        xb,yb=xb.to(DEVICE, non_blocking=False),yb.to(DEVICE, non_blocking=False)
        optimizer.zero_grad(set_to_none=True)
        loss=criterion(model(xb),yb)
        loss.backward(); optimizer.step()
        del loss
    train_rmse, train_mae, _, _ = evaluate_loader(model, train_loader)
    val_rmse, val_mae, _, _ = evaluate_loader(model, valid_loader)
    history.append([epoch,train_rmse,train_mae,val_rmse,val_mae])
    print(f"Epoch {epoch:02d} | Train RMSE {train_rmse:.4f} | Val RMSE {val_rmse:.4f} | Val MAE {val_mae:.4f}")
    if val_rmse < best_val_rmse:
        best_val_rmse=val_rmse; best_state=copy.deepcopy(model.state_dict()); patience=0
    else:
        patience += 1
        if patience >= PATIENCE:
            print("Early stopping."); break

model.load_state_dict(best_state)
history_df=pd.DataFrame(history,columns=["epoch","train_rmse","train_mae","val_rmse","val_mae"])
history_df

Epoch 01 | Train RMSE 3.3022 | Val RMSE 4.4700 | Val MAE 1.5154
Epoch 02 | Train RMSE 3.2645 | Val RMSE 4.4437 | Val MAE 1.5195
Epoch 03 | Train RMSE 3.2678 | Val RMSE 4.4283 | Val MAE 1.5503
Epoch 04 | Train RMSE 3.2453 | Val RMSE 4.4424 | Val MAE 1.4954
Epoch 05 | Train RMSE 3.2320 | Val RMSE 4.4294 | Val MAE 1.5155
Epoch 06 | Train RMSE 3.2072 | Val RMSE 4.3894 | Val MAE 1.5235
Epoch 07 | Train RMSE 3.2670 | Val RMSE 4.4066 | Val MAE 1.5512
Epoch 08 | Train RMSE 3.2035 | Val RMSE 4.3638 | Val MAE 1.5167
Epoch 09 | Train RMSE 3.1855 | Val RMSE 4.3049 | Val MAE 1.4904
Epoch 10 | Train RMSE 3.2090 | Val RMSE 4.3253 | Val MAE 1.5011
Epoch 11 | Train RMSE 3.1691 | Val RMSE 4.2919 | Val MAE 1.4873
Epoch 12 | Train RMSE 3.1301 | Val RMSE 4.2751 | Val MAE 1.5058
Epoch 13 | Train RMSE 3.1108 | Val RMSE 4.3235 | Val MAE 1.4987
Epoch 14 | Train RMSE 3.1131 | Val RMSE 4.3208 | Val MAE 1.4915
Epoch 15 | Train RMSE 3.0973 | Val RMSE 4.3214 | Val MAE 1.4834
Epoch 16 | Train RMSE 3.1323 | Val RMSE 

,epoch,train_rmse,train_mae,val_rmse,val_mae
0,1,3.302215,1.274784,4.470042,1.515394
1,2,3.264496,1.267827,4.443704,1.519512
2,3,3.267782,1.288947,4.428292,1.550288
3,4,3.245272,1.240954,4.442430,1.495372
4,5,3.231994,1.255911,4.429390,1.515475
5,6,3.207161,1.266329,4.389418,1.523478
6,7,3.267005,1.268162,4.406571,1.551230
7,8,3.203451,1.250838,4.363817,1.516654
8,9,3.185503,1.237406,4.304851,1.490434
9,10,3.208988,1.268225,4.325321,1.501146


## 5. 최종 Test 평가

Best validation checkpoint를 확정한 뒤 Test를 평가합니다.

In [8]:
test_rmse, test_mae, test_pred, test_actual = evaluate_loader(model, test_loader)
print("=== Final Test ===")
print(f"RMSE: {test_rmse:.4f}")
print(f"MAE : {test_mae:.4f}")

=== Final Test ===
RMSE: 3.4544
MAE : 1.2627


## 6. 시간에 따른 성능 변화

장기간 데이터에서 drift 여부를 살펴보기 위해 Test 결과를 연도/월별로 분해합니다.

In [9]:
test_result=pd.DataFrame({"target_time_utc":pd.to_datetime(ts_test),"actual":test_actual,"predicted":test_pred})
test_result["year"]=test_result.target_time_utc.dt.year
test_result["month"]=test_result.target_time_utc.dt.to_period("M").astype(str)
test_result["squared_error"]=(test_result.actual-test_result.predicted)**2
test_result["absolute_error"]=abs(test_result.actual-test_result.predicted)
yearly=test_result.groupby("year").agg(RMSE=("squared_error",lambda x:np.sqrt(x.mean())),MAE=("absolute_error","mean"),N=("absolute_error","size")).reset_index()
monthly=test_result.groupby("month").agg(RMSE=("squared_error",lambda x:np.sqrt(x.mean())),MAE=("absolute_error","mean"),N=("absolute_error","size")).reset_index()
display(yearly); display(monthly)

C:\Users\panteng\AppData\Local\Temp\ipykernel_49332\1515369035.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  test_result["month"]=test_result.target_time_utc.dt.to_period("M").astype(str)


,year,RMSE,MAE,N
0,2022,2.905561,1.215536,1575
1,2023,3.612145,1.277779,4933


,month,RMSE,MAE,N
0,2022-09,2.328949,1.714795,30
1,2022-10,3.640937,1.387374,546
2,2022-11,1.523827,1.006639,613
3,2022-12,3.403921,1.265414,386
4,2023-01,0.763146,0.543109,651
5,2023-02,1.036858,0.692295,308
6,2023-03,1.090530,0.715386,706
7,2023-04,5.647351,1.920348,641
8,2023-05,3.337783,1.135091,672
9,2023-06,1.626124,0.750056,595


In [10]:
torch.save({"model_state_dict":model.state_dict(),"feature_scaler":feature_scaler,"target_scaler":target_scaler,"input_shape":(24,3),"feature_order":features,"target":target_col},"factory_energy_lstm.pt")
print("baseline saved: factory_energy_lstm.pt")

import pickle as _pickle, sys as _sys
from pathlib import Path as _Path
_repo = _Path().resolve()
_repo = _repo if (_repo / "data" / "features.py").exists() else _repo.parent
if str(_repo) not in _sys.path:
    _sys.path.insert(0, str(_repo))
from data.features import EnergyScaler as _EnergyScaler
_bundle_scaler = _EnergyScaler()
_bundle_scaler.feature_scaler = feature_scaler
_bundle_scaler.target_scaler = target_scaler
with open("scaler.pkl", "wb") as _f:
    _pickle.dump(_bundle_scaler, _f)
print("baseline saved: scaler.pkl")

baseline saved: factory_energy_lstm.pt
